In [ ]:
import pandas as pd
import numpy as np
import commons as c
import plotly.express as px
import plotly.graph_objects as go

# Get datasets

In [ ]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)

In [ ]:
df_equiv['label'] = "Equivalent mutant"
df_normal['label'] = "Non-Equivalent mutant"

df = pd.concat([df_equiv, df_normal], ignore_index=True)
df['ideal_label'] = np.where(df['ideal_label'] == True, "Non-Equivalent mutant", "Equivalent mutant")
df['noisy_label'] = np.where(df['noisy_label'] == True, "Non-Equivalent mutant", "Equivalent mutant")

# Get Box Plots

In [ ]:
def print_box_plot(df, cat, output_folder, file_name, median_line=False):
    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column   
    
    # Create the box plot
    fig = px.box(
        df, 
        y="noisy_distance", 
        x=cat, 
        color="label", 
        category_orders={
            cat: cat_range,
            "label": ["Equivalent mutant", "Non-Equivalent mutant"]
        },
        title="Boxplot of Distance by noise model and program type",
        labels={"Characteristic": "Characteristic", 'noisy_distance': "Distance", 'label': "Legend"},
        points=False,
        boxmode="group"
    ) 
    
    # Compute medians for each category and true_label
    median_values = df.groupby([cat, "label"])['noisy_distance'].median().reset_index()
    
    if median_line:
        # Define colors matching the boxplot
        colors = {"Equivalent mutant": "blue", "Non-Equivalent mutant": "red"}
        
        for label in median_values["label"].unique():
            subset = median_values[median_values["label"] == label]
            fig.add_trace(go.Scatter(
                x=subset[cat], 
                y=subset["noisy_distance"], 
                mode='lines',
                name=f"Median - {label}",
                line=dict(color=colors[label], dash='dot') 
            ))
    
    # Adjust layout for better visualization
    fig.update_layout(
        xaxis=dict(tickmode="array", tickvals=cat_range)
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, f"Distance between the mutants and the oracle by {cat}", output_folder, file_name, yaxis_range=[0, 1])

In [ ]:
def category_plot(df, m):
    
    for hw in c.hardware:
        df_hw = df[df['hardware'] == hw]
        df_metric = df_hw[df_hw['metric'] == m]
        
        for cat in ['Qubits_number', 'gates', 'depth'] : 
            selected_columns = df_metric[[cat, 'label', 'ideal_distance', 'noisy_distance']]  
            file_name = f'{hw}_{cat}'
            output_folder = f'results/RQ2/RQ2_1/{m}'
            print_box_plot(selected_columns, cat, output_folder, file_name, True)
            
        for cat in ['Algorithm', 'Input_type', 'Output_type'] : 
            selected_columns = df_metric[[cat, 'label', 'ideal_distance', 'noisy_distance']]  
            file_name = f'{hw}_{cat}'
            output_folder = f'results/RQ2/RQ2_2/{m}'
            print_box_plot(selected_columns, cat, output_folder, file_name)
            
        for cat in ['Gate_type', 'Operator'] : 
            selected_columns = df_metric[[cat, 'label', 'ideal_distance', 'noisy_distance']]  
            file_name = f'{hw}_{cat}'
            output_folder = f'results/RQ2/RQ2_3/{m}'
            print_box_plot(selected_columns, cat, output_folder, file_name)
        
        for cat in ['Relative_position'] : 
            selected_columns = df_metric[[cat, 'label', 'ideal_distance', 'noisy_distance']]  
            file_name = f'{hw}_{cat}'
            output_folder = f'results/RQ2/RQ2_3/{m}'
            print_box_plot(selected_columns, cat, output_folder, file_name, True)

In [ ]:
m = "T"
category_plot(df, m)

m = "H"
category_plot(df, m)